In [2]:
!pip install transformers datasets torch scikit-learn pandas

In [4]:
import torch
import pandas as pd
from sklearn.metrics import accuracy_score
from datasets import load_dataset, Dataset
from transformers import (
    RobertaForSequenceClassification, DistilBertForSequenceClassification,
    RobertaTokenizer, DistilBertTokenizer,
    Trainer, TrainingArguments, pipeline
)

In [14]:

# Now you can safely tokenize using HuggingFace tokenizer
from transformers import AutoTokenizer

# Initialize your tokenizer (e.g., BERT tokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Function to tokenize the text
def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=512)

'''
# In-Domain: SST-2 (Stanford Sentiment Treebank)
sst2 = load_dataset("SST2_Dataset_Cleaned.csv")
train_df = pd.DataFrame({"text": sst2["train"]["review"], "label": sst2["train"]["label"]})
test_df = pd.DataFrame({"text": sst2["validation"]["review"], "label": sst2["validation"]["label"]})

# Cross-Domain: IMDB (Small subset for faster testing)
imdb = load_dataset("IMDB_Dataset_Cleaned.csv", split="test[:20%]") # Use 20% for demo
imdb_df = pd.DataFrame({"text": imdb["text"], "label": [1 if x == 'pos' else 0 for x in imdb["label"]]})
'''

# 🔧 Add this block right after loading
def untokenize(tokens):
    return " ".join(tokens) if isinstance(tokens, list) else tokens

# In-Domain: SST-2 (Stanford Sentiment Treebank)
# Load the cleaned SST-2 CSV file
sst2_df = pd.read_csv('SST2_Dataset_Cleaned.csv')

sst2_df["tokens"] = sst2_df["tokens"].apply(untokenize)

# Tokenize the "review" column before splitting
sst2_df["tokens"] = sst2_df["review"].apply(lambda x: tokenizer.tokenize(x))

# Split SST-2 into train and test (80% train, 20% test)
train_df = sst2_df[['review', 'label']].iloc[:int(len(sst2_df)*0.8)]
test_df = sst2_df[['review', 'label']].iloc[int(len(sst2_df)*0.8):]

# If you want to convert them into Hugging Face dataset format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Cross-Domain: IMDB (Small subset for faster testing)
# Load the cleaned IMDB CSV file
imdb_df = pd.read_csv('IMDB_Dataset_Cleaned.csv')

imdb_df["tokens"] = imdb_df["tokens"].apply(untokenize)

imdb_df["tokens"] = imdb_df["review"].apply(lambda x: tokenizer.tokenize(x))

# Convert the 'label' column to 1 if positive and 0 if negative
imdb_df['label'] = imdb_df['label'].apply(lambda x: 1 if x == 'pos' else 0)

# Use 20% of the data for testing
imdb_test_df = imdb_df.sample(frac=0.2, random_state=42)

# If you want to convert to Hugging Face dataset format
imdb_test_dataset = Dataset.from_pandas(imdb_test_df)

# You now have:
# - `train_dataset` and `test_dataset` for SST-2
# - `imdb_test_dataset` for IMDB (20% subset)


KeyboardInterrupt: 

In [ ]:
def create_fewshot_dataset(df, samples_per_class=8):
    return pd.concat([
        df[df["label"] == 0].sample(samples_per_class, random_state=42),
        df[df["label"] == 1].sample(samples_per_class, random_state=42)
    ])

fewshot_train = create_fewshot_dataset(train_df)
fewshot_dataset = Dataset.from_pandas(fewshot_train)
test_dataset = Dataset.from_pandas(test_df)
imdb_dataset = Dataset.from_pandas(imdb_df)

In [12]:
# RoBERTa
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# DistilBERT
distilbert_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return roberta_tokenizer(batch["review"], padding=True, truncation=True) # Same for DistilBERT

def dis_tokenize(batch):
    return distilbert_tokenizer(batch["review"], padding=True, truncation=True) # Same for DistilBERT

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

# RoBERTa Training
roberta_trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=fewshot_dataset.map(tokenize, batched=True),
    eval_dataset=test_dataset.map(tokenize, batched=True),
    compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))},
)
roberta_trainer.train()

# DistilBERT Training
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=fewshot_dataset.map(dis_tokenize, batched=True),
    eval_dataset=test_dataset.map(dis_tokenize, batched=True),
)
distilbert_trainer.train()

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/13645 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.694772,0.448516
2,No log,0.697979,0.448223
3,No log,0.698673,0.448223


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/13645 [00:00<?, ? examples/s]

IndexError: index out of range in self

In [ ]:
def evaluate_model(model, tokenizer, dataset):
    pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
    preds = pipe(dataset["review"])
    return accuracy_score(dataset["label"], [int(p["label"].split("_")[1]) for p in preds])

# Evaluate RoBERTa
roberta_imdb_acc = evaluate_model(roberta_model, roberta_tokenizer, imdb_dataset)
print(f"RoBERTa -> IMDB Accuracy: {roberta_imdb_acc:.2f}")

# Evaluate DistilBERT
distilbert_imdb_acc = evaluate_model(distilbert_model, distilbert_tokenizer, imdb_dataset)
print(f"DistilBERT -> IMDB Accuracy: {distilbert_imdb_acc:.2f}")

In [ ]:
class CoralTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels)

        # CORAL Loss (replace with IMDB features in practice)
        source_features = outputs.hidden_states[-1][:, 0, :] # CLS embeddings
        target_features = torch.randn_like(source_features) # Placeholder
        coral_loss = torch.norm(
            torch.cov(source_features.T) - torch.cov(target_features.T),
            p="fro"
        )
        return loss + 0.1 * coral_loss

# Retrain RoBERTa with CORAL
coral_trainer = CoralTrainer(
    model=roberta_model,
    args=training_args,
    train_dataset=fewshot_dataset.map(tokenize, batched=True),
)
coral_trainer.train()

In [ ]:
# Evaluate adapted model
adapted_imdb_acc = evaluate_model(coral_trainer.model, roberta_tokenizer, imdb_dataset)
print(f"RoBERTa + CORAL -> IMDB Accuracy: {adapted_imdb_acc:.2f}")

In [ ]:
print(f"RoBERTa: SST-2={roberta_trainer.evaluate(test_dataset)['eval_accuracy']:.2f}, IMDB={roberta_imdb_acc:.2f} -> {adapted_imdb_acc:.2f} (CORAL)")
print(f"DistilBERT: SST-2={distilbert_trainer.evaluate(test_dataset)['eval_accuracy']:.2f}, IMDB={distilbert_imdb_acc:.2f}")